# Очистка данных о звонках (Calls)

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import re

import help_130625_dam as h

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Загрузка данных

In [2]:
DATA_PATH = os.path.join('..', 'Sources', 'Calls (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'calls_clean.pkl')

# Читаем ID как строки, это предотвращает округление 19-значных чисел при загрузке
df = pd.read_excel(DATA_PATH, dtype={'Id': str, 'CONTACTID': str})

df.columns = [h.to_snake(c) for c in df.columns]
n_before = len(df)

print(f'Форма: {df.shape}')
h.descr_df(df, include='all', show_sample_rows=True)

Форма: (95874, 11)


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,str,95874,0,95874,5805028000000805001,5805028000000768006,5805028000000764027,NaN,NaN,NaN,NaN
1,call_start_time,str,95874,0,68445,30.06.2023 08:43,30.06.2023 08:46,30.06.2023 08:59,NaN,NaN,NaN,NaN
2,call_owner_name,str,95874,0,33,John Doe,John Doe,John Doe,NaN,NaN,NaN,NaN
3,contactid,str,91941,3933,15214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,call_type,str,95874,0,3,Inbound,Outbound,Outbound,NaN,NaN,NaN,NaN
5,call_duration_in_seconds,float64,95791,83,2619,171.00,28.00,24.00,0.00,164.98,8.00,7625.00
6,call_status,str,95874,0,11,Received,Attended Dialled,Attended Dialled,NaN,NaN,NaN,NaN
7,dialled_number,float64,0,95874,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,outgoing_call_status,str,86875,8999,4,NaN,Completed,Completed,NaN,NaN,NaN,NaN
9,scheduled_in_crm,float64,86875,8999,2,NaN,0.00,0.00,0.00,0.00,0.00,1.00


In [3]:
# Дубликаты по уникальному ID звонка
id_duplicates = df.duplicated(subset=['id']).sum()
print(f"Количество дубликатов по 'id': {id_duplicates}")

if id_duplicates > 0:
    df = df.drop_duplicates(subset=['id'], keep='first')
    print(f"Удалены технические дубликаты по 'id'. Осталось строк: {len(df)}")

# Содержательные дубликаты (одно время, один менеджер, один контакт и одна длительность)
# Если длительность разная — это могут быть параллельные вызовы или особенности телефонии, их оставляем.
cols_to_check = ['call_start_time', 'call_owner_name', 'contactid', 'call_duration_in_seconds']
content_duplicates = df.duplicated(subset=cols_to_check, keep=False).sum()
print(f"Количество строк с полными содержательными дубликатами: {content_duplicates}")

if content_duplicates > 0:
    # Удаляем только те, где совпадает абсолютно всё, включая длительность
    df = df.drop_duplicates(subset=cols_to_check, keep='first')
    print(f"Удалено содержательных дубликатов. Осталось строк: {len(df)}")

# Анализ содержимого удаляемых колонок
print("Уникальные значения в 'Outgoing Call Status':")
print(df['outgoing_call_status'].value_counts(dropna=False))

# print("\nСравнение с 'Call Status' для Outbound звонков:")
# display(df[df['call_type'] == 'Outbound'][['call_status', 'outgoing_call_status']].head(10))

Количество дубликатов по 'id': 0
Количество строк с полными содержательными дубликатами: 6416
Удалено содержательных дубликатов. Осталось строк: 92599
Уникальные значения в 'Outgoing Call Status':
outgoing_call_status
Completed    83717
NaN           8804
Overdue         56
Cancelled       19
Scheduled        3
Name: count, dtype: int64


In [4]:
# Анализ связи между запланированными звонками и их статусом
print("Распределение 'Scheduled in CRM':")
print(df['scheduled_in_crm'].value_counts(dropna=False))

print("\nСтатусы звонков, которые были запланированы (True):")
print(df[df['scheduled_in_crm'] == True]['call_status'].value_counts().head(10))

print("\nСтатусы звонков, которые НЕ были запланированы (False):")
print(df[df['scheduled_in_crm'] == False]['call_status'].value_counts().head(10))

Распределение 'Scheduled in CRM':
scheduled_in_crm
0.00    83661
NaN      8804
1.00      134
Name: count, dtype: int64

Статусы звонков, которые были запланированы (True):
call_status
Overdue                       56
Cancelled                     19
Scheduled Attended Delay      19
Scheduled Unattended Delay    17
Scheduled Attended            14
Scheduled Unattended           6
Scheduled                      3
Name: count, dtype: int64

Статусы звонков, которые НЕ были запланированы (False):
call_status
Attended Dialled      69521
Unattended Dialled    14140
Name: count, dtype: int64


На основе предварительного анализа мы планируем удалить:
1. **`outgoing_call_status`**: Для Outbound звонков значения дублируют `call_status` или не несут дополнительной аналитической ценности.
2. **`scheduled_in_crm`**: Поле содержит техническую информацию о том, был ли звонок запланирован. Запланировано 134 звонка из 95874.
3. **`tag`**: Колонка практически не заполнена.
4. **`dialled_number`**: олонка практически не заполнена.

## Первичный анализ и исправление типов

In [ ]:
df['id'] = pd.array([h.str_to_int64(v) for v in df['id']], dtype='Int64')
df['contactid'] = pd.array([h.str_to_int64(v) for v in df['contactid']], dtype='Int64')

# Пропущенные contactid заменяем на -1, это позволит сохранить тип данных 
# и не потерять строки с неизвестным контактом
df['contactid'] = df['contactid'].fillna(-1)

# Исправление дат
date_cols = [col for col in df.columns if 'time' in col or 'date' in col]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# Базовая очистка и оптимизация типов
df['call_duration_in_seconds'] = df['call_duration_in_seconds'].fillna(0).astype('int32')

# Текстовые столбцы переводим в category
str_cols = ['call_owner_name', 'call_type', 'call_status']
for col in str_cols:
    df[col] = df[col].astype('category')

# Удаление неиспользуемых столбцов
cols_to_drop = ['dialled_number', 'tag', 'outgoing_call_status', 'scheduled_in_crm']
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f'contactid dtype: {df["contactid"].dtype} | NaN: {df["contactid"].isna().sum()}')
print(f'Количество звонков с неизвестным contactid (-1): {(df["contactid"] == -1).sum()}')

contactid dtype: Int64 | NaN: 0
Количество звонков с неизвестным contactid (-1): 3799


In [ ]:
# Проверка на перекрывающиеся звонки у одного менеджера
# Рассчитаем время окончания звонка
df['call_end_time'] = df['call_start_time'] + pd.to_timedelta(df['call_duration_in_seconds'], unit='s')

# Сортируем для проверки перекрытий
df_sorted = df.sort_values(['call_owner_name', 'call_start_time'])

# Сдвигаем время окончания предыдущего звонка того же менеджера
df_sorted['prev_call_end'] = df_sorted.groupby('call_owner_name')['call_end_time'].shift(1)

# Условие перекрытия
overlapping = df_sorted[df_sorted['call_start_time'] < df_sorted['prev_call_end']].copy()

if len(overlapping) > 0:
    print(f"Обнаружено {len(overlapping)} строк с пересечением времени.")
    overlapping['overlap_seconds'] = (overlapping['prev_call_end'] - overlapping['call_start_time']).dt.total_seconds()
    print("\nРаспределение величины пересечения (секунды):")
    print(overlapping['overlap_seconds'].describe())
else:
    print("Пересекающихся звонков не обнаружено.")


Обнаружено 7913 строк с пересечением времени.

Распределение величины пересечения (секунды):
count   7913.00
mean      72.76
std      269.26
min        1.00
25%        5.00
50%        7.00
75%       14.00
max     7140.00
Name: overlap_seconds, dtype: float64


### Выводы по пересекающимся звонкам:
В данных обнаружено **9107** случаев временного перекрытия звонков у одного и того же менеджера.

**Возможные причины:**
1. **Технические особенности CRM (75% случаев):** Большинство накладок составляют менее 13 секунд. Это может быть связано с тем, что система начинает запись нового звонка или автодозвон до того, как менеджер закроет карточку предыдущего клиента.
2. **Параллельные линии:** Использование гарнитур с поддержкой нескольких вызовов или работа в нескольких вкладках CRM одновременно.
3. Система может инициировать звонок заранее, чтобы минимизировать простой менеджера.
4. **Ошибки логирования данных:** Неточное фиксирование времени завершения (`Call End Time`) при обрыве связи или программных сбоях.
5. **Аномалии (длинные пересечения):** Одиночные случаи накладок в несколько десятков минут могут указывать на "зависшие" сессии звонков, которые не были корректно завершены в системе.

*Данные аномалии не критичны для общего анализа воронки, так как составляют менее 10% данных и в большинстве своем являются короткими техническими накладками.*


In [ ]:
# Флаг успешного дозвона: статус «в трубку взяли» + есть длительность
df['is_successful'] = (
    df['call_status'].isin(['Attended Dialled', 'Received']) &
    (df['call_duration_in_seconds'] > 0)
)
print(f'is_successful: {df["is_successful"].sum():,} успешных из {len(df):,} ({df["is_successful"].mean()*100:.1f}%)')

is_successful: 72,590 успешных из 92,599 (78.4%)


In [ ]:
h.descr_df(df, include=['all'], show_sample_rows=False)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Минимум,Среднее,Медиана,Максимум
0,id,Int64,92599,0,92599,5805028000000764027,5805028000031551488.00,5805028000032459776.00,5805028000056912329
1,call_start_time,datetime64[us],92599,0,68445,<NA>,<NA>,<NA>,<NA>
2,call_owner_name,category,92599,0,33,<NA>,<NA>,<NA>,<NA>
3,contactid,Int64,92599,0,15215,-1,5566868825822548992.00,5805028000024195072.00,5805028000056892055
4,call_type,category,92599,0,3,<NA>,<NA>,<NA>,<NA>
5,call_duration_in_seconds,int32,92599,0,2619,0,170.22,9.00,7625
6,call_status,category,92599,0,11,<NA>,<NA>,<NA>,<NA>
7,call_end_time,datetime64[us],92599,0,90604,<NA>,<NA>,<NA>,<NA>
8,is_successful,bool,92599,0,2,<NA>,<NA>,<NA>,<NA>


In [ ]:
MAPPING_PATH = os.path.join('..', 'data', 'cleaned', 'contact_mapping.pkl')

# Применяем маппинг дублей контактов (сформирован в 01_cleaning_contacts)
if os.path.exists(MAPPING_PATH):
    contact_mapping = pd.read_pickle(MAPPING_PATH)
    affected = df['contactid'].isin(contact_mapping.keys()).sum()
    df['contactid'] = df['contactid'].replace(contact_mapping.to_dict())
    print(f"Маппинг контактов применён: {affected} звонков перепривязаны к мастер-контактам.")
else:
    print("Файл contact_mapping.pkl не найден. Сначала выполните 01_cleaning_contacts.")

# Сохранение очищенных данных
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)
df.to_excel(OUT_PATH.replace('.pkl', '.xlsx'))

summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Уникальных звонков (id)',
        'Диапазон времени звонков',
        'Уникальных владельцев',
        'Кол-во неизвестных контактов (-1)',
        'Пропуски в финальном DF'
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        df['id'].nunique(),
        f'{df["call_start_time"].min().date()} → {df["call_start_time"].max().date()}',
        df['call_owner_name'].nunique(),
        (df['contactid'] == -1).sum(),
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {OUT_PATH}')
display(pd.DataFrame(summary_data))

Маппинг контактов применён: 110 звонков перепривязаны к мастер-контактам.
Сохранено: ..\data\cleaned\calls_clean.pkl


,Метрика,Значение
0,Строк исходно,95874
1,Строк после очистки,92599
2,Удалено дубликатов,3275
3,Уникальных звонков (id),92599
4,Диапазон времени звонков,2023-06-30 → 2024-06-21
5,Уникальных владельцев,33
6,Кол-во неизвестных контактов (-1),3799
7,Пропуски в финальном DF,0


## Описательная статистика

In [ ]:
# 1. Сводная статистика для числовых полей
numeric_cols = ['call_duration_in_seconds']

desc_stats = df[numeric_cols].describe().T
desc_stats['median'] = df[numeric_cols].median()
desc_stats['range'] = desc_stats['max'] - desc_stats['min']

# Моду считаем отдельно (берем первую)
desc_stats['mode'] = df[numeric_cols].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

print("--- Сводная статистика числовых полей ---")
display(desc_stats[['mean', 'median', 'mode', 'min', 'max', 'range', 'std']].round(2))

# 2. Анализ категориальных полей
cat_cols = ['call_owner_name', 'call_type', 'call_status', 'is_successful']

print("\n--- Анализ категориальных полей (Топ-10 значений) ---")
for col in cat_cols:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        pct = (df[col].value_counts(normalize=True, dropna=False) * 100).round(1)
        
        stat_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pct}).head(10)
        print(f"\nПоле: {col}")
        display(stat_df)

--- Сводная статистика числовых полей ---


,mean,median,mode,min,max,range,std
call_duration_in_seconds,170.22,9.00,0,0.00,7625.00,7625.00,406.83



--- Анализ категориальных полей (Топ-10 значений) ---

Поле: call_owner_name


,Count,Percentage (%)
call_owner_name,,
Yara Edwards,8530,9.20
Julia Nelson,7211,7.80
Ian Miller,7026,7.60
Charlie Davis,6942,7.50
Diana Evans,6713,7.20
Ulysses Adams,5960,6.40
Amy Green,5574,6.00
Victor Barnes,5361,5.80
Kevin Parker,5357,5.80



Поле: call_type


,Count,Percentage (%)
call_type,,
Outbound,83795,90.50
Missed,5734,6.20
Inbound,3070,3.30



Поле: call_status


,Count,Percentage (%)
call_status,,
Attended Dialled,69521,75.10
Unattended Dialled,14140,15.30
Missed,5735,6.20
Received,3069,3.30
Overdue,56,0.10
Cancelled,19,0.00
Scheduled Attended Delay,19,0.00
Scheduled Unattended Delay,17,0.00
Scheduled Attended,14,0.00



Поле: is_successful


,Count,Percentage (%)
is_successful,,
True,72590,78.40
False,20009,21.60


## Описание датасета

**Источник:** `Calls (Done).xlsx` — выгрузка данных о звонках из CRM  
**Назначение:** анализ активности менеджеров, оценка качества обработки лидов и расчет метрик дозвона.

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID записи о звонке |
| `contactid` | `int64` | ID связанного контакта (связь с `contacts.id`). При загрузке указан `dtype=str`, что сохранило точность 19-значных ID. |
| `call_start_time` | `datetime` | Дата и время начала звонка |
| `call_duration_in_seconds` | `int32` | Длительность разговора в секундах |
| `is_successful` | `bool` | **Флаг дозвона:** True, если длительность > 0 и статус 'Attended Dialled' или 'Received' |
| `call_owner_name` | `category` | Менеджер, совершивший или принявший звонок |
| `call_type` | `category` | Тип звонка (Inbound/Outbound) |
| `call_status` | `category` | Результат (Completed, Missed и др.) |

### Особенности данных и очистка
- **Точность ID:** Благодаря чтению `CONTACTID` как строки из Excel, мы избежали потери точности последних цифр (проблема 15-значного лимита float64 в Excel).
- **Дубликаты:** Удалено 6416 полных "содержательных" дубликатов (совпадение времени, менеджера, контакта и длительности). Это записи, возникшие из-за технических сбоев CRM.
- **Пропуски в `contactid`:** Записи без привязки к контакту заполнены значением `-1`. Это позволяет сохранить звонки для анализа нагрузки менеджеров, не теряя их при анализе связей.
- **Пересекающиеся звонки:** Обнаружено ~10% технических накладок во времени (в основном менее 13 секунд), что характерно для работы автодозвона и систем логирования.

**Ключевые связи:**
- `contactid` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `call_start_time` → используется для расчета Speed to Lead в общих отчетах.
- `is_successful` → основной фильтр для оценки эффективности коммуникаций.